In [1]:
 # usual imports
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# NLP-specific imports

# CountVectorizer => the most simple vectorizing method for text data
from sklearn.feature_extraction.text import CountVectorizer

# import the multinomial Naive-Bayes (supports more than 2 outcomes too)
from sklearn.naive_bayes import MultinomialNB

In [2]:
# load the data and see the contents
# we need to clean this dataset as here is too many columns.
# out tasget variable is "research_type" lets's check it out.
df = pd.read_csv("biomedical_research_abstracts_2024_2026.csv")
df.tail()

,pmid,title,abstract,abstract_words,journal,pub_year,pub_month,pub_month_num,month_year,doi,authors_count,country,research_type,keywords,major_topic,language,open_access
126827,37015487,The Risks of Ranking: Revisiting Graphical Per...,Graphical perception studies typically measure...,209,IEEE transactions on visualization and compute...,2024,Mar,3,Mar-2024,10.1109/TVCG.2022.3226463,8,Unknown,"Journal Article; Research Support, U.S. Gov't,...",Humans; Individuality; Bayes Theorem; Computer...,Individuality,eng,False
126828,37015450,Personalized Audio-Driven 3D Facial Animation ...,We present a learning-based approach for gener...,198,IEEE transactions on visualization and compute...,2024,Mar,3,Mar-2024,10.1109/TVCG.2022.3230541,4,Unknown,"Journal Article; Research Support, U.S. Gov't,...",Humans; Computer Graphics; Face; Speech,Computer Graphics,eng,False
126829,36711949,Functional characterization of the language ne...,How do polyglots-individuals who speak five or...,198,bioRxiv : the preprint server for biology,2024,Jan,1,Jan-2024,10.1101/2023.01.19.524657,8,MA 02139,Preprint; Journal Article,NaN,NaN,eng,True
126830,36455093,Marching Windows: Scalable Mesh Generation for...,Volumetric data abounds in medical imaging and...,231,IEEE transactions on visualization and compute...,2024,Mar,3,Mar-2024,10.1109/TVCG.2022.3225526,7,Unknown,Journal Article,NaN,NaN,eng,True
126831,36346866,A Survey of Smooth Vector Graphics: Recent Adv...,The field of smooth vector graphics explores t...,195,IEEE transactions on visualization and compute...,2024,Mar,3,Mar-2024,10.1109/TVCG.2022.3220575,2,Unknown,Journal Article,NaN,NaN,eng,False


In [3]:
# here and further sometimes I used AI:
# select target column
target = df["research_type"]

# split labels by ";"
split_labels = target.str.split(";")

# flatten lists into rows
exploded_labels = split_labels.explode()

# remove extra spaces
clean_labels = exploded_labels.str.strip()

# count frequency
counts = clean_labels.value_counts()

# get top 8
top_8 = counts.head(8)

# print results
print("Top Research Types:")
for label, count in top_8.items():
    print(f"{label}: {count}")

Top Research Types:
Journal Article: 126347
Research Support, Non-U.S. Gov't: 11774
Review: 10234
Case Reports: 4968
Systematic Review: 2531
Research Support, N.I.H., Extramural: 2187
Randomized Controlled Trial: 1659
Meta-Analysis: 1494


In [4]:
# I will map the target variable into 5 categories: RCT, Meta-Analysis, Review, Case Report, and Other.
# Mapping function to categorize research types
def categorize_research_type(research_type):
    research_type_lower = str(research_type).lower().strip()
    
    if "rct" in research_type_lower or "randomized" in research_type_lower:
        return "RCT"
    elif "meta" in research_type_lower or "systematic" in research_type_lower:
        return "Meta-Analysis"
    elif "review" in research_type_lower:
        return "Review"
    elif "case" in research_type_lower:
        return "Case Report"
    else:
        return "Other"

# Clean and categorize the research types
df['research_type_cleaned'] = df['research_type'].str.split(';').apply(
    lambda x: [categorize_research_type(label.strip()) for label in x]
)

# Explode and count the cleaned categories
cleaned_categories = df['research_type_cleaned'].explode().value_counts()
print("── Cleaned Research Type Categories ──")
print(cleaned_categories)

── Cleaned Research Type Categories ──
research_type_cleaned
Other            147435
Review            10656
Case Report        4969
Meta-Analysis      4103
RCT                1748
Name: count, dtype: int64


In [5]:
# Now we have not ballanced target variable, let's drop rows with only "Other" category. 
df.head()

,pmid,title,abstract,abstract_words,journal,pub_year,pub_month,pub_month_num,month_year,doi,authors_count,country,research_type,keywords,major_topic,language,open_access,research_type_cleaned
0,41869676,"The genome sequence of the bloodfluke planorb,...",We present a genome assembly from an individua...,57,Wellcome open research,2024,Unknown,0,Unknown-2024,10.12688/wellcomeopenres.22819.2,7,UK,Journal Article,NaN,NaN,eng,True,[Other]
1,41841041,An exploratory study of critical incidents wit...,Leadership changes within public organizations...,250,F1000Research,2024,Unknown,0,Unknown-2024,10.12688/f1000research.142942.2,4,Indonesia,Journal Article,Leadership; Humans; Anxiety; Organizational In...,Leadership,eng,True,[Other]
2,41797731,From Mary Shelley to Netflix: a Pan-European p...,Scientific knowledge of the human brain has ca...,244,Frontiers in neuroscience,2024,Unknown,0,Unknown-2024,10.3389/fnins.2024.1278640,2,Spain,Journal Article,NaN,NaN,eng,True,[Other]
3,41726542,Improving electronic health record processing ...,Large language models (LLMs) excel in natural ...,134,AMIA ... Annual Symposium proceedings. AMIA Sy...,2024,Unknown,0,Unknown-2024,NaN,4,USA,Journal Article,Natural Language Processing; Electronic Health...,Natural Language Processing,eng,True,[Other]
4,41726541,Developing a User-Centered Mobile Application ...,This study is part of the OsteoPorotic fracTur...,136,AMIA ... Annual Symposium proceedings. AMIA Sy...,2024,Unknown,0,Unknown-2024,NaN,13,USA,Journal Article,Mobile Applications; Humans; Skilled Nursing F...,Mobile Applications,eng,True,[Other]


In [6]:
# Filter out rows with "Other" category AND remove "Other" from lists
# with help of ChatGPT
df['research_type_cleaned'] = df['research_type_cleaned'].apply(
    lambda x: [cat for cat in x if cat != "Other"]
)
df = df[df["research_type_cleaned"].apply(lambda x: len(x) > 0)].reset_index(drop=True)

In [7]:
df.describe()

,pmid,abstract_words,pub_year,pub_month_num,authors_count
count,1.994500e+04,19945.000000,19945.000000,19945.000000,19945.000000
mean,3.908495e+07,206.763650,2024.261670,2.842216,6.089847
std,1.296982e+06,76.292325,0.550924,3.206839,4.620924
min,3.399038e+07,50.000000,2024.000000,0.000000,1.000000
25%,3.822727e+07,152.000000,2024.000000,1.000000,4.000000
50%,3.829278e+07,201.000000,2024.000000,1.000000,5.000000
75%,3.989691e+07,249.000000,2024.000000,4.000000,7.000000
max,4.188182e+07,996.000000,2026.000000,12.000000,159.000000


In [8]:
df.head()

,pmid,title,abstract,abstract_words,journal,pub_year,pub_month,pub_month_num,month_year,doi,authors_count,country,research_type,keywords,major_topic,language,open_access,research_type_cleaned
0,41726485,Earlier ICU Transfer after CONCERN Early Warni...,Early recognition and timely escalation of car...,137,AMIA ... Annual Symposium proceedings. AMIA Sy...,2024,Unknown,0,Unknown-2024,NaN,9,NY,Journal Article; Multicenter Study; Randomized...,Humans; Sepsis; Intensive Care Units; Hospital...,Sepsis,eng,True,[RCT]
1,41726474,Cultural Prompting Improves the Empathy and Cu...,Large Language Model (LLM)-based conversationa...,149,AMIA ... Annual Symposium proceedings. AMIA Sy...,2024,Unknown,0,Unknown-2024,NaN,7,WA,Journal Article; Randomized Controlled Trial,Humans; Empathy; Cultural Competency; Female; ...,Empathy,eng,True,[RCT]
2,41726454,Computational Use of Patient-Provider Secure M...,Secure messaging (SM) between patients and pro...,148,AMIA ... Annual Symposium proceedings. AMIA Sy...,2024,Unknown,0,Unknown-2024,NaN,5,United States,Journal Article; Systematic Review,Humans; Computer Security; Communication,Computer Security,eng,True,[Meta-Analysis]
3,41700312,Recycling of collagen from solid tannery waste...,This study explores the innovative potential o...,227,F1000Research,2024,Unknown,0,Unknown-2024,10.12688/f1000research.155450.2,4,Ecuador,Journal Article; Review,Collagen; Recycling; Adhesives; Tanning; Indus...,Collagen,eng,True,[Review]
4,41542283,Linking evidence for targeted blood biomarkers...,With improvements in acute stroke treatment an...,300,Frontiers in stroke,2024,Unknown,0,Unknown-2024,10.3389/fstro.2024.1491542,5,United States,Journal Article; Review,NaN,NaN,eng,True,[Review]


In [9]:
df['research_type_cleaned'].explode().value_counts()


research_type_cleaned
Review           10656
Case Report       4969
Meta-Analysis     4103
RCT               1748
Name: count, dtype: int64

In [10]:
# Print research_type_cleaned
print(df['research_type_cleaned'])

0                                 [RCT]
1                                 [RCT]
2                       [Meta-Analysis]
3                              [Review]
4                              [Review]
                      ...              
19940                          [Review]
19941    [Meta-Analysis, Meta-Analysis]
19942           [Meta-Analysis, Review]
19943                   [Meta-Analysis]
19944                          [Review]
Name: research_type_cleaned, Length: 19945, dtype: object


In [11]:
# now we can see that "Review" category has 10656 rows, which is much more than other categories.
# To avoid bias in the model,I will limit the number of "Review" rows to 4000 by random sampling.
# Create a function to limit Review category
def limit_review(df, max_reviews=4000):
    review_rows = df[df['research_type_cleaned'].apply(lambda x: 'Review' in x)]
    other_rows = df[~df['research_type_cleaned'].apply(lambda x: 'Review' in x)]
    
    if len(review_rows) > max_reviews:
        review_rows = review_rows.sample(n=max_reviews, random_state=42)
    
    return pd.concat([review_rows, other_rows]).reset_index(drop=True)

df = limit_review(df, max_reviews=4000)

In [ ]:
# somehow we have 4004 rpws of "Review" category, but it's ok.
df['research_type_cleaned'].explode().value_counts()

research_type_cleaned
Case Report      4817
Meta-Analysis    4045
Review           4004
RCT              1747
Name: count, dtype: int64